In [1]:
## read file names in a dictionary
import os
from natsort import natsorted
root = '/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/'
Conditions = ['35Hz12kN','37.5Hz11kN/','40Hz10kN/']
files = {}

for c in range(len(Conditions)): # 3 conditions
    for b in range(5): # 5 bearings
        bearing = str('Bearing'+str(c+1)+'_'+str(b+1))
        print(bearing)
        root_bearing = os.path.join(root, Conditions[c],bearing)
        file1 = []
        for dirname, dirnames, filenames in os.walk(root_bearing):
            # print path to all subdirectories first.
            for filename in filenames:             
                if '#' in filename:
                    continue 
                if 'checkpoints' in dirname:
                    continue 
                file1.append(os.path.join(dirname, filename))
        file2 = natsorted(file1)
        print(len(file2))
        files[bearing] = file2
        print(files[bearing][0])
        print(files[bearing][-1])


Bearing1_1
123
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_1/1.csv
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_1/123.csv
Bearing1_2
161
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_2/1.csv
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_2/161.csv
Bearing1_3
158
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_3/1.csv
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_3/158.csv
Bearing1_4
122
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_4/1.csv
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_4/122.csv
Bearing1_5
52
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_5/1.csv
/home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/35Hz12kN/Bearing1_5/52.csv
Bearing2_1
491
/home/jingyi/Jy/Gearbo

## FFT

In [17]:
## bearing parameter
d = 7.92 # mm, diameter of rolling elements
Z = 8 # number of rolling elements
De = 39.8 # mm, diameter of the outer race
Di = 29.3 # mm, diameter of inner race
Dm = 34.55 # mm, bearing mean diameter (= pitch diameter) (De+Di)/2
alpha = 0 # contact angle of bearing

In [18]:
## feature order
import math
Fif_o = Z/2*(1+d/Dm*math.cos(alpha))
Fof_o = Z/2*(1-d/Dm*math.cos(alpha))
Fbf_o = Dm/d/2*(1-(d/Dm*math.cos(alpha))**2)
Fcf_o = 1/2*(1-d/Dm*math.cos(alpha))
Fcf_z_o = Z/2*(1-d/Dm*math.cos(alpha))
# print(Fif,Fof,Fbf,Fcf)
signatures = 'feature order: Fif_o(' + str(round(Fif_o,2))+ ') Fof_o(' + str(round(Fof_o,2)) + ') Fbf_o(' + str(round(Fbf_o,2)) + ') Fcf_o(' + str(round(Fcf_o,2))+') Fcf_z_o(' + str(round(Fcf_z_o,2)) + ')' 
print(signatures)

feature order: Fif_o(4.92) Fof_o(3.08) Fbf_o(2.07) Fcf_o(0.39) Fcf_z_o(3.08)


In [46]:
## feature ofer for cage fault except Fcf_z_o
Fcf_zi_o = [item * Fcf_o for item in list(range(1,Z))] # replace Z+1 to Z as Fcf_z_o is excepted
print('Fcf_zi_o: '+str([round(item,2) for item in Fcf_zi_o]))

Fcf_zi_o: [0.39, 0.77, 1.16, 1.54, 1.93, 2.31, 2.7]


In [54]:
## order = Freq/Freq_ref
T = 1/25600 # Sample Spacing, 1/25.6k
def order(array, Fr):    
    N = len(array) # Number of samplepoints
    f = scipy.fftpack.fft(array.values,axis=0)
    f2 = 2.0/N * np.abs(f[:N//2,:])
    Freqf = np.linspace(0.0, 1.0/(2.0*T), N//2) # half of length, the maximum frequency the half of sampling frequency
    Orderf = Freqf/Fr
    return Orderf, f2

In [93]:
import os
import pandas as pd
import numpy as np
import scipy.fftpack
from multiprocessing import Pool

# Function to process a single file
def process_file(keys):
    try:

        ## get reference frequency
        if '1_' in keys:
            Fr = 35
        elif '2_' in keys:
            Fr = 37.5
        elif '3_' in keys:
            Fr = 40

        ## columns 2(vertical and horizontal)*5 = 10
        ## Fr_o(1), Fif_o(4.92) Fof_o(3.08) Fbf_o(2.07) Fcf_o
        ## Fcf_o = Max(Fcf_zi_o: [0.39, 0.77, 1.16, 1.54, 1.93, 2.31, 2.7])
        Orders = pd.DataFrame(columns=['Fr_o(Horizontal)','Fr_o(Vertical)',
                                       'Fif_o(Horizontal)','Fif_o(Vertical)','Fof_o(Horizontal)','Fof_o(Vertical)',
                                       'Fbf_o(Horizontal)','Fbf_o(Vertical)','Fcf_o(Horizontal)','Fcf_o(Vertical)'])        
        
        for i in range(len(files[keys])):
            bearing = pd.read_csv(files[keys][i])
            Orderf, amps = order(bearing, Fr)            

            ## get feature order index
            if i == 0:
                features = [Orderf[Orderf>=1][0],Orderf[Orderf>=Fif_o][0],Orderf[Orderf>=Fof_o][0],
                            Orderf[Orderf>=Fbf_o][0]] # Fr_o(1), Fif_o(4.92) Fof_o(3.08) Fbf_o(2.07)
                features_Fcf = [Orderf[Orderf>=item][0] for item in Fcf_zi_o] # Fcf_zi_o: [0.39, 0.77, 1.16, 1.54, 1.93, 2.31, 2.7]
                indices = [np.argwhere(Orderf == item)[0][0] for item in features]  
                indices_Fcf = [np.argwhere(Orderf == item)[0][0] for item in features_Fcf] 

            ## get amplitude of feature orders
            for j in range(len(indices)):
                idx = indices[j]
                idx_extended = list(range(max(idx-5,0),min(idx+5,len(Orderf))))
                Orders.loc[i,Orders.columns[[j*2,j*2+1]]] = [max(amps[idx_extended,0]),max(amps[idx_extended,1])] # horizontal, vertical
            # for cage
            idx_extended = list()
            for j in range(len(indices_Fcf)):
                idx = indices_Fcf[j]
                idx_extended = idx_extended+list(range(max(idx-5,0),min(idx+5,len(Orderf))))
            Orders.loc[i,['Fcf_o(Horizontal)','Fcf_o(Vertical)']] = [max(amps[idx_extended,0]),max(amps[idx_extended,1])] # horizontal, vertical

        output_name = os.path.join(root, keys+f'_Orders.csv')
        
        Orders.to_csv(output_name, index=True)
        print(f"Processed: {output_name}")
        
    except Exception as e:
        print(f"Error processing {keys}: {e}")

# Main execution
if __name__ == "__main__":

    with Pool(processes=os.cpu_count()) as pool:
        pool.map(process_file, files.keys())
        

Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing2_4_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_5_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing3_5_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_4_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_1_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_3_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing2_2_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_2_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing2_5_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing3_3_Orders.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU

## RMS

In [14]:
import os
import pandas as pd
import numpy as np
import numpy_rms
from multiprocessing import Pool
# Function to process a single file
def process_file(keys): 
    try:
        rms = pd.DataFrame(columns=['Horizontal RMS','Vertical RMS'])

        for i in range(len(files[keys])):
            df = pd.read_csv(files[keys][i])
            rms.loc[i,:]=[numpy_rms.rms(df[df.columns[0]].to_numpy())[0],numpy_rms.rms(df[df.columns[1]].to_numpy())[0]]

        output_name = os.path.join(root, keys+f'_RMS.csv')
        rms.to_csv(output_name, index=False)            
        print(f"Processed: {output_name}")
        
    except Exception as e:
        print(f"Error processing {keys}: {e}")

# Main execution
if __name__ == "__main__":
    with Pool(processes=os.cpu_count()) as pool:
        pool.map(process_file, files.keys())

Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing2_4_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_5_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing3_5_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_4_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_1_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_2_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing2_2_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_3_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing2_5_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing3_3_RMS.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing

### Kurtosis

In [22]:
import os
import pandas as pd
import numpy as np
from scipy.stats import kurtosis
from multiprocessing import Pool
# Function to process a single file
def process_file(keys): 
    try:
        k = pd.DataFrame(columns=['Horizontal Kurtosis','Vertical Kurtosis'])

        for i in range(len(files[keys])):
            df = pd.read_csv(files[keys][i])
            k.loc[i,:]=[kurtosis(df[df.columns[0]].to_numpy()),kurtosis(df[df.columns[1]].to_numpy())]

        output_name = os.path.join(root, keys+f'_Kurtosis.csv')
        k.to_csv(output_name, index=False)            
        print(f"Processed: {output_name}")
        
    except Exception as e:
        print(f"Error processing {keys}: {e}")

# Main execution
if __name__ == "__main__":
    with Pool(processes=os.cpu_count()) as pool:
        pool.map(process_file, files.keys())

Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing2_4_Kurtosis.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_5_Kurtosis.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing3_5_Kurtosis.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_4_Kurtosis.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_1_Kurtosis.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing2_2_Kurtosis.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_3_Kurtosis.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing1_2_Kurtosis.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing2_5_Kurtosis.csv
Processed: /home/jingyi/Jy/Gearbox Monitoring/XJTU-XY_BearingRUL/XJTU-SY/Bearing3_3_Kurtosis.csv
Processed: /home/jingyi/Jy/Gea